# APA Calling

## Aim
The purpose of this notebook is to call APA-based information (PDUI) based on [DAPARS2 method](https://github.com/3UTR/DaPars2).

## Methods

**Timing:** Runtime varies by dataset size and compute resources. For the toy chr22 MWE dataset, most steps complete in under 10 minutes on a standard HPC node.

In [ ]:
%preview ../../images/apa_calling.png

### 3'UTR Reference

* _gtf2bed12.py_ : Covert gtf to bed format (Source from in-house codes from Li Lab: https://github.com/Xu-Dong/Exon_Intron_Extractor/blob/main/scripts/gtf2bed12.py)  

* _DaPars_Extract_Anno.py_ : extract the 3UTR regions in bed formats from the whole genome bed (Source from Dapars 2: https://github.com/3UTR/DaPars2/blob/master/src/DaPars_Extract_Anno.py)

### Call WIG data from transcriptome BAM files

Using bedtools or rsem-bam2wig, for RSEM based alignment

### Config files Generation
* _Python 3_ loops to read line by line the sum of reads coverage of all chromosome.

### Dapars2 Main Function

* _Dapars2_Multi_Sample.py_: use the least sqaures methods to calculate the usage of long isoforms (https://github.com/3UTR/DaPars2/blob/master/src/Dapars2_Multi_Sample.py)  
    
    Note: this part of code have been modified from source to deal with some formatting discrepancy in wig file

### Impute missing values in Dapars result

KNN using `impute` R package.

## Input

- A list of transcriptome level BAM files, eg generated by RSEM
- The 3'UTR annotation reference file

If you do not have 3'UTR annotation file, please generate it first. Input to this step is the transcriptome level gene feature file in `GTF` format that [we previously prepared](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/reference_data.html).

## Output

*  Dapars config files
*  PUDI (Raw) information saved in txt
*  PDUI (Imputed) information saved in txt. This is recommended for further analysis. 

## Steps

**Step 1.** Generate the 3'UTR reference regions from a GTF annotation:

In [ ]:
sos run pipeline/apa_calling.ipynb UTR_reference \
    --cwd output/apa \
    --hg-gtf output/apa/chr22.gtf

**Step 2.** Convert the transcriptome BAM files into per-base coverage (wig) and read-depth (flagstat) files:

In [ ]:
sos run pipeline/apa_calling.ipynb bam2tools \
    --cwd output/apa \
    --bam-dir output/rnaseq/bam

**Step 3.** Generate the DaPars2 sample configuration and mapping files:

In [ ]:
sos run pipeline/apa_calling.ipynb APAconfig \
    --cwd output/apa \
    --bfile output/apa/wig \
    --annotation output/apa/chr22_3UTR.bed

**Step 4.** Run DaPars2 to calculate the PDUI matrix per chromosome:

In [ ]:
sos run pipeline/apa_calling.ipynb APAmain \
    --cwd output/apa \
    --chrlist chr22 \
    --dapars-path code/molecular_phenotypes/calling/apa/Dapars2_Multi_Sample.py

## Command interface

In [ ]:
sos run pipeline/apa_calling.ipynb -h

## Workflow implementation

In [ ]:
[global]
parameter: walltime = '400h'
parameter: mem = '200G'
parameter: ncore = 16
# the output directory for generated files
parameter: cwd = path("output")
# Number of threads
parameter: numThreads = 8
parameter: job_size = 1
parameter: container = ''


### **Step 0.** Generate 3'UTR regions based on GTF

In [ ]:
# Generate the 3UTR region according to the gtf file
[UTR_reference]
# gtf file
parameter: hg_gtf = path
# Directory holding the bundled DaPars helper scripts (gtf2bed12.py, DaPars_Extract_Anno.py)
parameter: apa_src = path("code/molecular_phenotypes/calling/apa")
input: hg_gtf
output: f'{cwd}/{_input:bn}.bed', 
        f'{cwd}/{_input:bn}.transcript_to_geneName.txt',
        f'{cwd}/{_input:bn}_3UTR.bed'
bash: expand = '${ }'
    module load python2/2.7.16
    python2 ${apa_src}/gtf2bed12.py --gtf ${_input} --out ${cwd}
    mv ${cwd}/gene_annotation.bed ${_output[0]}
    mv ${cwd}/transcript_to_geneName.txt ${_output[1]}
    python2 ${apa_src}/DaPars_Extract_Anno.py -b ${_output[0]} -s ${_output[1]} -o ${_output[2]}

### **Step 1.** Generate WIG coverage and flagstat files from BAM files

In [ ]:
[bam2tools]
# Directory containing transcriptome-level BAM files (*.Aligned.sortedByCoord.out.bam)
parameter: bam_dir = path
# Output directory for the per-sample wig and flagstat files
parameter: wig_dir = path(f"{cwd}/wig")
input: [str(x) for x in path(bam_dir).iterdir() if str(x).endswith("Aligned.sortedByCoord.out.bam")], group_by = 1
sample_id = _input.name.split('.')[0]
output: f'{wig_dir}/{sample_id}.wig', f'{wig_dir}/{sample_id}.flagstat'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads
python: expand = "${ }"
    import os
    os.makedirs("${wig_dir}", exist_ok = True)
    # 1) per-base coverage track for DaPars2
    os.system("bedtools genomecov -ibam ${_input} -bga -split -trackline > ${_output[0]}")
    # 2) sequencing depth (total reads) used to build the Dapars2 config
    os.system("samtools flagstat --threads ${numThreads} ${_input} > ${_output[1]}")


### **Step 2.** Generate the DaPars2 configuration and sample mapping files

In [ ]:
# Generate configuration file
[APAconfig]
parameter: bfile = path
parameter: annotation = path
parameter: job_size = 1
# Default parameters for Dapars2:
parameter: least_pass_coverage_percentage = 0.3
parameter: coverage_threshold = 10
output: [f'{cwd}/sample_mapping_files.txt',f'{cwd}/sample_configuration_file.txt']
task: trunk_workers = 1, trunk_size = 1, walltime = walltime, mem = mem, cores = ncore
python3: expand = "${ }"
    import re
    import os
    target_all_sample = os.listdir("${bfile}")
    target_all_sample = list(filter(lambda v: re.match('.*wig$', v), target_all_sample))
    target_all_sample = ["${bfile}" + "/" + w for w in target_all_sample]
    

    def extract_total_reads(input_flagstat_file):
        num_line = 0
        total_reads = '-1'
        #print input_flagstat_file
        for line in open(input_flagstat_file,'r'):
            num_line += 1
            if num_line == 1:  # total reads is on line 1 ('N + M in total') in current samtools
                total_reads = line.strip().split(' ')[0]
                break
        return total_reads

    #print(target_all_sample)
    print("INFO: Total",len(target_all_sample),"samples found in provided dirctory!")
    # Total depth file:
    mapping_file = open("${_output[0]}", "w")
    for current_sample in target_all_sample:
        flag = current_sample.split(".")[0] + ".flagstat"
        current_sample_total_depth = extract_total_reads(flag)
        field_out = [current_sample, str(current_sample_total_depth)]
        mapping_file.writelines('\t'.join(field_out) + '\n')

        print("Coverage of sample ", current_sample, ": ", current_sample_total_depth)
    mapping_file.close()

    # Configuration file:

    config_file = open(${_output[1]:r},"w")
    config_file.writelines(f"Annotated_3UTR=${annotation}\n")
    config_file.writelines( "Aligned_Wig_files=%s\n" % ",".join(target_all_sample))
    config_file.writelines(f"Output_directory=${cwd}/apa \n")
    config_file.writelines(f"Output_result_file=Dapars_result\n")
    config_file.writelines(f"Least_pass_coverage_percentage=${least_pass_coverage_percentage}\n")
    config_file.writelines( "Coverage_threshold=${coverage_threshold}\n")
    config_file.writelines( "Num_Threads=${numThreads}\n")
    config_file.writelines(f"sequencing_depth_file=${_output[0]}") 
    config_file.close()    

### **Step 3.** Run DaPars2 main to calculate PDUIs

## Anticipated Results

The pipeline produces output files in the `output/` subdirectory named after the workflow step. Verify success by checking that output files exist and are non-empty. See the **Output** section above for the expected file names and formats.

In [ ]:
# Call Dapars2 multi_chromosome
[APAmain]
parameter: chr_prefix = False
parameter: chrlist = list
# Path to the bundled DaPars2 multi-sample script
parameter: dapars_path = path(f"{cwd:a}/../code/molecular_phenotypes/calling/apa/Dapars2_Multi_Sample.py")
input: for_each = 'chrlist'
output: [f'{cwd}/apa_{x}/Dapars_result_result_temp.{x}.txt' for x in chrlist]
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = ncore
bash: expand = True
    module load python2/2.7.16
    python2 {dapars_path} {cwd}/sample_configuration_file.txt {_chrlist} {"F" if chr_prefix else "T"}
